<a href="https://colab.research.google.com/github/togich/big-data-course/blob/main/Hadoop%20HDFS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Установка Java и Hadoop

In [2]:
!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk ssh pdsh 2>/dev/null

# !wget -q --show-progress https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
# !wget -q --show-progress https://dlcdn.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!ls -lh hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz -C /content/

import os
os.environ['HADOOP_HOME'] = '/content/hadoop-3.3.6'
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = f"{os.environ['HADOOP_HOME']}/bin:{os.environ['HADOOP_HOME']}/sbin:{os.environ['PATH']}"

print("✅ Hadoop установлен")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
-rw-r--r-- 1 root root 697M Jun 25  2023 hadoop-3.3.6.tar.gz
✅ Hadoop установлен


1.1 Проверка установки

In [3]:
# 1. Java установлена?
!java -version 2>&1 | head -1

# 2. Hadoop распакован?
!ls /content/hadoop-3.3.6/bin/hdfs

# 3. Размер архива ~696 МБ?
!ls -lh hadoop-3.3.6.tar.gz | awk '{print $5}'

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
/content/hadoop-3.3.6/bin/hdfs
697M


Ожидаемый результат:

Java: openjdk version "11..." или нет ошибок

/content/hadoop-3.3.6/bin/hdfs существует

Размер: ~696M

#2. Настройка Hadoop

In [4]:
import os
HADOOP_HOME = '/content/hadoop-3.3.6'

with open(f'{HADOOP_HOME}/etc/hadoop/hadoop-env.sh', 'a') as f:
    f.write(f'\nexport JAVA_HOME={os.environ["JAVA_HOME"]}\n')
    f.write('export HDFS_NAMENODE_USER=root\n')
    f.write('export HDFS_DATANODE_USER=root\n')
    f.write('export HDFS_SECONDARYNAMENODE_USER=root\n')

core_site = """<?xml version="1.0" encoding="UTF-8"?>
<configuration>
    <property><name>fs.defaultFS</name><value>hdfs://localhost:9001</value></property>
</configuration>"""
with open(f'{HADOOP_HOME}/etc/hadoop/core-site.xml', 'w') as f:
    f.write(core_site)

hdfs_site = """<?xml version="1.0" encoding="UTF-8"?>
<configuration>
    <property><name>dfs.replication</name><value>1</value></property>
    <property><name>dfs.namenode.name.dir</name><value>file:/content/hadoop_data/namenode</value></property>
    <property><name>dfs.datanode.data.dir</name><value>file:/content/hadoop_data/datanode</value></property>
</configuration>"""
with open(f'{HADOOP_HOME}/etc/hadoop/hdfs-site.xml', 'w') as f:
    f.write(hdfs_site)

!rm -rf /content/hadoop_data
!mkdir -p /content/hadoop_data/namenode /content/hadoop_data/datanode

print("✅ Настройка завершена")

✅ Настройка завершена


2.1 Контрольная проверка

In [5]:
# 1. JAVA_HOME прописан в hadoop-env.sh?
!grep "JAVA_HOME" /content/hadoop-3.3.6/etc/hadoop/hadoop-env.sh

# 2. core-site.xml содержит правильный адрес?
!cat /content/hadoop-3.3.6/etc/hadoop/core-site.xml | grep -A1 defaultFS

# 3. hdfs-site.xml содержит replication=1?
!cat /content/hadoop-3.3.6/etc/hadoop/hdfs-site.xml | grep -A1 replication

# 4. Папки для данных созданы?
!ls -d /content/hadoop_data/namenode /content/hadoop_data/datanode

#  JAVA_HOME=/usr/java/testing hdfs dfs -ls
# Technically, the only required environment variable is JAVA_HOME.
# export JAVA_HOME=
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
    <property><name>fs.defaultFS</name><value>hdfs://localhost:9001</value></property>
</configuration>
    <property><name>dfs.replication</name><value>1</value></property>
    <property><name>dfs.namenode.name.dir</name><value>file:/content/hadoop_data/namenode</value></property>
/content/hadoop_data/datanode  /content/hadoop_data/namenode


Ожидаемый результат:

export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64

"<value>hdfs://localhost:9001</value>"

"<value>1</value>"

Обе папки существуют

#3. Форматирование и запуск HDFS

In [8]:
!rm -rf /content/hadoop_data
!mkdir -p /content/hadoop_data/namenode /content/hadoop_data/datanode
!ls -la /content/hadoop_data/

# Форматируем
!/content/hadoop-3.3.6/bin/hdfs namenode -format -force 2>&1 | tail -20

total 16
drwxr-xr-x 4 root root 4096 Oct  6 07:46 .
drwxr-xr-x 1 root root 4096 Oct  6 07:46 ..
drwxr-xr-x 2 root root 4096 Oct  6 07:46 datanode
drwxr-xr-x 2 root root 4096 Oct  6 07:46 namenode
2026-10-06 07:46:50,069 INFO metrics.TopMetrics: NNTop conf: dfs.namenode.top.num.users = 10
2026-10-06 07:46:50,069 INFO metrics.TopMetrics: NNTop conf: dfs.namenode.top.windows.minutes = 1,5,25
2026-10-06 07:46:50,075 INFO namenode.FSNamesystem: Retry cache on namenode is enabled
2026-10-06 07:46:50,075 INFO namenode.FSNamesystem: Retry cache will use 0.03 of total heap and retry cache entry expiry time is 600000 millis
2026-10-06 07:46:50,077 INFO util.GSet: Computing capacity for map NameNodeRetryCache
2026-10-06 07:46:50,077 INFO util.GSet: VM type       = 64-bit
2026-10-06 07:46:50,078 INFO util.GSet: 0.029999999329447746% max memory 3.2 GB = 996.6 KB
2026-10-06 07:46:50,078 INFO util.GSet: capacity      = 2^17 = 131072 entries
2026-10-06 07:46:50,110 INFO namenode.FSImage: Allocated new

In [9]:
# Останавливаем всё, что могло запуститься
!pkill -f namenode 2>/dev/null
!pkill -f datanode 2>/dev/null
!pkill -f secondarynamenode 2>/dev/null
!sleep 2

# Запускаем демоны напрямую (без SSH, без pdsh, без start-dfs.sh)
!/content/hadoop-3.3.6/bin/hdfs --daemon start namenode
!/content/hadoop-3.3.6/bin/hdfs --daemon start datanode

# Ждём
!sleep 5

# Проверяем
!jps

^C
^C
^C
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
44625 Jps
44483 NameNode
44551 DataNode


3.1 Проверка запуска

In [10]:
# 1. Процессы запущены?
!jps | grep -E "NameNode|DataNode|SecondaryNameNode"

# 2. HDFS отвечает?
!/content/hadoop-3.3.6/bin/hdfs dfs -ls /

# 3. Порт 9001 слушается?
!ss -tlnp | grep 9001

44483 NameNode
44551 DataNode
LISTEN 0      256        127.0.0.1:9001       0.0.0.0:*    users:(("java",pid=44483,fd=349))      


Ожидаемый результат:

Два процесса: NameNode, DataNode

ls / не выдаёт ошибку (пустой вывод — норма)

Порт 9001 в состоянии LISTEN

#4. Функция hdfs() — Задание 1

In [11]:
import subprocess

HDFS_BIN = '/content/hadoop-3.3.6/bin/hdfs'

def hdfs(cmd):
    """Выполняет команду hdfs dfs и возвращает stdout"""
    result = subprocess.run(
        f"{HDFS_BIN} dfs {cmd}",
        shell=True, capture_output=True, text=True
    )
    if result.stderr and 'WARN' not in result.stderr:
        print("⚠️", result.stderr[:300])
    return result.stdout

print(hdfs("-ls /"))

4.1 Контрольная проверка



In [12]:
# 1. Функция возвращает строку?
result = hdfs("-ls /")
print(f"Тип результата: {type(result)}")
assert isinstance(result, str), "hdfs() должна возвращать строку"

# 2. Ошибок нет?
print("✅ Функция hdfs() работает")

# 3. Сравнение с локальным ls
print("\n--- HDFS ---")
print(hdfs("-ls /"))
print("\n--- Локальная ФС ---")
!ls /

Тип результата: <class 'str'>
✅ Функция hdfs() работает

--- HDFS ---


--- Локальная ФС ---
bin		   home		      opt		 sbin.usr-is-merged
bin.usr-is-merged  kaggle	      proc		 srv
boot		   lib		      python-apt	 sys
content		   lib64	      python-apt.tar.xz  tmp
datalab		   lib.usr-is-merged  root		 tools
dev		   media	      run		 usr
etc		   mnt		      sbin		 var


Ожидаемый результат:

hdfs() возвращает str

Содержимое HDFS / и локального / разное — это доказывает, что это разные ФС

Ответ на контрольный вопрос: hdfs dfs -ls — команда HDFS (внутри распределённой ФС), ls — команда локальной файловой системы.

#5. Проверка данных

In [13]:
import os
DATA_DIR = "/content/mimic-iv-clinical-database-demo-2.2"

print("Содержимое:", os.listdir(DATA_DIR))
for folder in ["hosp", "icu"]:
    p = f"{DATA_DIR}/{folder}"
    if os.path.exists(p):
        print(f"✅ {folder}/ — {len(os.listdir(p))} файлов")

Содержимое: ['icu', 'demo_subject_id.csv', 'SHA256SUMS.txt', 'LICENSE.txt', '.ipynb_checkpoints', 'README.txt', 'hosp']
✅ hosp/ — 22 файлов
✅ icu/ — 9 файлов


5.1 Контрольная проверка

In [14]:
# 1. Все обязательные файлы на месте?
for f in ["demo_subject_id.csv", "README.txt", "SHA256SUMS.txt", "LICENSE.txt"]:
    p = f"{DATA_DIR}/{f}"
    print(f"{'✅' if os.path.exists(p) else '❌'} {f}")

# 2. Количество файлов совпадает с SHA256SUMS.txt?
with open(f"{DATA_DIR}/SHA256SUMS.txt") as f:
    lines = [l for l in f if l.strip()]
hosp_in_checksums = [l for l in lines if "hosp/" in l]
icu_in_checksums = [l for l in lines if "icu/" in l]
print(f"\nВ SHA256SUMS.txt: hosp/ = {len(hosp_in_checksums)}, icu/ = {len(icu_in_checksums)}")

# 3. Реальное количество файлов
print(f"Реально: hosp/ = {len(os.listdir(f'{DATA_DIR}/hosp'))}, icu/ = {len(os.listdir(f'{DATA_DIR}/icu'))}")

✅ demo_subject_id.csv
✅ README.txt
✅ SHA256SUMS.txt
✅ LICENSE.txt

В SHA256SUMS.txt: hosp/ = 22, icu/ = 9
Реально: hosp/ = 22, icu/ = 9


Ожидаемый результат:

4 метафайла на месте

Количество в SHA256SUMS.txt совпадает с реальным (или отличается — тогда запишите расхождение)

#6. Задание 2. Дерево MIMIC-IV в HDFS

In [15]:
import os

DATA_DIR = "/content/mimic-iv-clinical-database-demo-2.2"

# === 1. Создаём структуру каталогов ===
hdfs("-mkdir -p /mimic/demo/meta")
hdfs("-mkdir -p /mimic/demo/raw/hosp")
hdfs("-mkdir -p /mimic/demo/raw/icu")
hdfs("-mkdir -p /mimic/demo/processed")
hdfs("-mkdir -p /mimic/demo/results")

# === 2. Загружаем метаданные (4 файла) ===
for f in ["demo_subject_id.csv", "README.txt", "SHA256SUMS.txt", "LICENSE.txt"]:
    path = f"{DATA_DIR}/{f}"
    if os.path.exists(path):
        hdfs(f'-put "{path}" /mimic/demo/meta/')
        print(f"✅ {f}")
    else:
        print(f"❌ {f} — не найден локально")

# === 3. Загружаем hosp (22 файлов) ===
hdfs(f'-put "{DATA_DIR}/hosp/"* /mimic/demo/raw/hosp/')

# === 4. Загружаем icu (9 файлов) ===
hdfs(f'-put "{DATA_DIR}/icu/"* /mimic/demo/raw/icu/')

# === 5. Проверяем результат ===
print("\n=== Дерево /mimic/demo ===")
print(hdfs("-ls -R /mimic/demo"))

✅ demo_subject_id.csv
✅ README.txt
✅ SHA256SUMS.txt
✅ LICENSE.txt

=== Дерево /mimic/demo ===
drwxr-xr-x   - root supergroup          0 2026-10-06 07:51 /mimic/demo/meta
-rw-r--r--   1 root supergroup      25815 2026-10-06 07:51 /mimic/demo/meta/LICENSE.txt
-rw-r--r--   1 root supergroup        978 2026-10-06 07:51 /mimic/demo/meta/README.txt
-rw-r--r--   1 root supergroup       2966 2026-10-06 07:51 /mimic/demo/meta/SHA256SUMS.txt
-rw-r--r--   1 root supergroup        911 2026-10-06 07:51 /mimic/demo/meta/demo_subject_id.csv
drwxr-xr-x   - root supergroup          0 2026-10-06 07:51 /mimic/demo/processed
drwxr-xr-x   - root supergroup          0 2026-10-06 07:51 /mimic/demo/raw
drwxr-xr-x   - root supergroup          0 2026-10-06 07:51 /mimic/demo/raw/hosp
-rw-r--r--   1 root supergroup      11072 2026-10-06 07:51 /mimic/demo/raw/hosp/admissions.csv.gz
-rw-r--r--   1 root supergroup     510070 2026-10-06 07:51 /mimic/demo/raw/hosp/d_hcpcs.csv.gz
-rw-r--r--   1 root supergroup    18135

6.1 Контрольная проверка

In [16]:
# Каталоги
print("=== Каталоги ===")
for d in ["meta", "raw/hosp", "raw/icu", "processed", "results"]:
    result = hdfs(f"-test -d /mimic/demo/{d} && echo OK || echo NO")
    print(f"{'✅' if 'OK' in result else '❌'} /mimic/demo/{d}")

# Количество файлов
print("\n=== Файлы ===")
def count_files(path):
    lines = hdfs(f"-ls {path}").split("\n")
    return len([l for l in lines if l.strip() and not l.startswith("Found")])

print(f"meta/: {count_files('/mimic/demo/meta')} файлов (ожидается 4)")
print(f"hosp/: {count_files('/mimic/demo/raw/hosp')} файлов (ожидается 22)")
print(f"icu/:  {count_files('/mimic/demo/raw/icu')} файлов (ожидается 9)")

=== Каталоги ===
✅ /mimic/demo/meta
✅ /mimic/demo/raw/hosp
✅ /mimic/demo/raw/icu
✅ /mimic/demo/processed
✅ /mimic/demo/results

=== Файлы ===
meta/: 4 файлов (ожидается 4)
hosp/: 22 файлов (ожидается 22)
icu/:  9 файлов (ожидается 9)


Ожидаемый результат:

5 каталогов созданы

hosp/ = 23 файлов, icu/ = 10 файлов

В meta/ — 4 файла

Ответы на контрольные вопросы:

В hosp/ — 22 файлов.

В icu/ — 9 файлов.

demo_subject_id.csv — метаданные, поэтому в meta/.

#7. Задание 3. Блоки, реплики, целостность

In [17]:
import subprocess

HDFS_BIN = '/content/hadoop-3.3.6/bin/hdfs'

def hdfs(cmd):
    """Универсальная: -fsck → hdfs fsck, остальное → hdfs dfs"""
    s = cmd.strip()
    if s.startswith("-fsck"):
        full = f"{HDFS_BIN} fsck {s[5:].strip()}"
    else:
        full = f"{HDFS_BIN} dfs {s}"
    r = subprocess.run(full, shell=True, capture_output=True, text=True)
    if r.stderr and 'WARN' not in r.stderr:
        print("⚠️", r.stderr[:300])
    return r.stdout


# 1. Блоки и реплики
print("=== chartevents.csv.gz ===")
print(hdfs("-fsck /mimic/demo/raw/icu/chartevents.csv.gz -files -blocks -locations"))

print("\n=== patients.csv.gz ===")
print(hdfs("-fsck /mimic/demo/raw/hosp/patients.csv.gz -files -blocks -locations"))

# 2. Скачиваем файлы для SHA256
hdfs("-get -f /mimic/demo/raw/hosp/patients.csv.gz ./check_patients.csv.gz")
hdfs("-get -f /mimic/demo/raw/hosp/labevents.csv.gz ./check_labevents.csv.gz")

# 3. Считаем SHA256
!sha256sum ./check_patients.csv.gz
!sha256sum ./check_labevents.csv.gz

=== chartevents.csv.gz ===
⚠️ Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&locations=1&path=%2Fmimic%2Fdemo%2Fraw%2Ficu%2Fchartevents.csv.gz

FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path /mimic/demo/raw/icu/chartevents.csv.gz at Tue Oct 06 07:52:59 UTC 2026

/mimic/demo/raw/icu/chartevents.csv.gz 5549389 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1642924831-172.28.0.12-1791272810100:blk_1073741852_1028 len=5549389 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-ff026a17-4247-4b48-af89-eb3c96a83741,DISK]]


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	5549389 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 5549389 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-replicated blocks:	0 (0.0 %)
 Mis-replicated blocks:		0 (0.0 %)
 Default replication factor:	1
 Average block replicat

7.1 Контрольная проверка

In [19]:
import re
import hashlib

# 1. fsck: patients
fsck_out = hdfs("-fsck /mimic/demo/raw/hosp/patients.csv.gz -files -blocks")
print("patients.csv.gz:")
print(f"  Статус:  {'✅ HEALTHY' if 'HEALTHY' in fsck_out else '❌ НЕ HEALTHY'}")
print(f"  Блоков:  {len(re.findall(r'blk_', fsck_out))}")
print(f"  Реплик:  {re.findall(r'repl=(\d+)', fsck_out)}")

# 2. fsck: chartevents
fsck_chart = hdfs("-fsck /mimic/demo/raw/icu/chartevents.csv.gz -files -blocks")
print("\nchartevents.csv.gz:")
print(f"  Статус:  {'✅ HEALTHY' if 'HEALTHY' in fsck_chart else '❌ НЕ HEALTHY'}")
print(f"  Реплик:  {re.findall(r'repl=(\d+)', fsck_chart)}")

# 3. SHA256
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

actual_p = sha256("./check_patients.csv.gz")
expect_p = "59dcc6cd8450ddf03ad32cc7d76e41caa4c9633541715987f590c74835cf6fae"

actual_l = sha256("./check_labevents.csv.gz")
expect_l = "d51a4cf1ea63245abe7d544b0d50a928e9ee2ec5eb9a5ce85500aed0bd6c6dfc"

print("\nSHA256:")
print(f"  patients:   {'✅' if actual_p == expect_p else '❌'}")
print(f"  labevents:  {'✅' if actual_l == expect_l else '❌'}")

# 4. Итог
print("\n" + "=" * 40)
checks = {
    "fsck patients": "HEALTHY" in fsck_out,
    "fsck chartevents": "HEALTHY" in fsck_chart,
    "SHA256 patients": actual_p == expect_p,
    "SHA256 labevents": actual_l == expect_l,
}
for name, ok in checks.items():
    print(f"  {'✅' if ok else '❌'} {name}")
print(f"\nИтого: {sum(checks.values())}/{len(checks)}")

⚠️ Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&path=%2Fmimic%2Fdemo%2Fraw%2Fhosp%2Fpatients.csv.gz

patients.csv.gz:
  Статус:  ✅ HEALTHY
  Блоков:  1
  Реплик:  ['1']
⚠️ Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&path=%2Fmimic%2Fdemo%2Fraw%2Ficu%2Fchartevents.csv.gz


chartevents.csv.gz:
  Статус:  ✅ HEALTHY
  Реплик:  ['1']

SHA256:
  patients:   ✅
  labevents:  ✅

  ✅ fsck patients
  ✅ fsck chartevents
  ✅ SHA256 patients
  ✅ SHA256 labevents

Итого: 4/4


Ожидаемый результат:

Статус: HEALTHY

Реплик: ['1'] (в нашем Colab одна машина)

Оба SHA256: ✅ СОВПАЛ

Ответы на вопросы:

Если сумма совпала — файл не повреждён. Это доказательство целостности.

При отказе DataNode HDFS восстановит блок из реплики.

128 МБ — баланс между метаданными NameNode и параллелизмом.

#8. Задание 4. Автоматизация на Python

In [20]:
import hashlib
import os

def compute_sha256(file_path):
    sha = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            sha.update(chunk)
    return sha.hexdigest()

def resolve_hdfs_path(path):
    """
    Преобразует путь из SHA256SUMS.txt в реальный путь HDFS.
    - LICENSE.txt, README.txt, demo_subject_id.csv, SHA256SUMS.txt → meta/
    - hosp/... → raw/hosp/...
    - icu/...  → raw/icu/...
    """
    if path.startswith("hosp/"):
        return f"/mimic/demo/raw/{path}"
    elif path.startswith("icu/"):
        return f"/mimic/demo/raw/{path}"
    else:
        # метаданные: LICENSE.txt, README.txt, demo_subject_id.csv, SHA256SUMS.txt
        return f"/mimic/demo/meta/{path}"

def verify_all(checksums_file, hdfs_base="/mimic/demo"):
    expected = {}
    with open(checksums_file) as f:
        for line in f:
            line = line.strip()
            if line:
                sha, path = line.split(maxsplit=1)
                expected[path.strip()] = sha

    results = []
    for path, exp_sha in expected.items():
        # Преобразуем путь
        hdfs_path = resolve_hdfs_path(path)

        # Уникальное локальное имя
        filename = path.replace("/", "_")
        local = f"/tmp/{filename}"

        if os.path.exists(local):
            os.remove(local)

        # Скачиваем
        hdfs(f"-get {hdfs_path} {local}")

        if not os.path.exists(local):
            print(f"⚠️  SKIP  {path} (нет в HDFS: {hdfs_path})")
            results.append((path, None))
            continue

        actual = compute_sha256(local)
        results.append((path, actual == exp_sha))
    return results


# === Запуск ===
checksums = f"{DATA_DIR}/SHA256SUMS.txt"
results = verify_all(checksums)

ok_count = 0
skip_count = 0
for path, ok in results:
    if ok is None:
        skip_count += 1
    elif ok:
        print(f"✅ OK    {path}")
        ok_count += 1
    else:
        print(f"❌ FAIL  {path}")

print(f"\nИтого: ✅ {ok_count}, ❌ {len(results) - ok_count - skip_count}, ⚠️ {skip_count} пропущено")

✅ OK    LICENSE.txt
✅ OK    README.txt
✅ OK    demo_subject_id.csv
✅ OK    hosp/admissions.csv.gz
✅ OK    hosp/d_hcpcs.csv.gz
✅ OK    hosp/d_icd_diagnoses.csv.gz
✅ OK    hosp/d_icd_procedures.csv.gz
✅ OK    hosp/d_labitems.csv.gz
✅ OK    hosp/diagnoses_icd.csv.gz
✅ OK    hosp/drgcodes.csv.gz
✅ OK    hosp/emar.csv.gz
✅ OK    hosp/emar_detail.csv.gz
✅ OK    hosp/hcpcsevents.csv.gz
✅ OK    hosp/labevents.csv.gz
✅ OK    hosp/microbiologyevents.csv.gz
✅ OK    hosp/omr.csv.gz
✅ OK    hosp/patients.csv.gz
✅ OK    hosp/pharmacy.csv.gz
✅ OK    hosp/poe.csv.gz
✅ OK    hosp/poe_detail.csv.gz
✅ OK    hosp/prescriptions.csv.gz
✅ OK    hosp/procedures_icd.csv.gz
✅ OK    hosp/provider.csv.gz
✅ OK    hosp/services.csv.gz
✅ OK    hosp/transfers.csv.gz
✅ OK    icu/caregiver.csv.gz
✅ OK    icu/chartevents.csv.gz
✅ OK    icu/d_items.csv.gz
✅ OK    icu/datetimeevents.csv.gz
✅ OK    icu/icustays.csv.gz
✅ OK    icu/ingredientevents.csv.gz
✅ OK    icu/inputevents.csv.gz
✅ OK    icu/outputevents.csv.gz
✅ OK   

8.1 Контрольная проверка

In [21]:
# 1. Сколько файлов проверено?
print(f"Проверено: {len(results)}")
print(f"Успешно: {ok_count}")
print(f"Провалено: {len(results) - ok_count}")

# 2. Если есть провалы — какие?
failed = [p for p, ok in results if not ok]
if failed:
    print("\n❌ Провалены:")
    for p in failed:
        print(f"  {p}")
else:
    print("\n✅ Все файлы целые")

# 3. Сравнение с количеством в SHA256SUMS.txt
with open(checksums) as f:
    total_in_file = len([l for l in f if l.strip()])
print(f"\nСтрок в SHA256SUMS.txt: {total_in_file}")
print(f"Проверено: {len(results)}")
assert total_in_file == len(results), "Количество не совпадает!"
print("✅ Количество совпадает")

Проверено: 34
Успешно: 34
Провалено: 0

✅ Все файлы целые

Строк в SHA256SUMS.txt: 34
Проверено: 34
✅ Количество совпадает


Ожидаемый результат:

Проверено: 34

Успешно: 34

Провалено: 0

Ответы на вопросы:

Автоматизация нужна, потому что 34 файла вручную проверять долго и легко ошибиться.

Если сумма не совпала — файл повреждён, нужно перезагрузить.

#9. Задание 5. Демонстрация для защиты

In [22]:
print("=== 1. Дерево /mimic/demo ===")
print(hdfs("-ls -R /mimic/demo"))

print("\n=== 2. Блоки и реплики ===")
print(hdfs("-fsck /mimic/demo/raw/hosp/patients.csv.gz -files -blocks -locations"))

print("\n=== 3. SHA256 ===")
!sha256sum ./check_patients.csv.gz

print("\n=== 4. Python-скрипт ===")
print(f"Проверено файлов: {len(results)}, успешно: {ok_count}")

=== 1. Дерево /mimic/demo ===
drwxr-xr-x   - root supergroup          0 2026-10-06 07:51 /mimic/demo/meta
-rw-r--r--   1 root supergroup      25815 2026-10-06 07:51 /mimic/demo/meta/LICENSE.txt
-rw-r--r--   1 root supergroup        978 2026-10-06 07:51 /mimic/demo/meta/README.txt
-rw-r--r--   1 root supergroup       2966 2026-10-06 07:51 /mimic/demo/meta/SHA256SUMS.txt
-rw-r--r--   1 root supergroup        911 2026-10-06 07:51 /mimic/demo/meta/demo_subject_id.csv
drwxr-xr-x   - root supergroup          0 2026-10-06 07:51 /mimic/demo/processed
drwxr-xr-x   - root supergroup          0 2026-10-06 07:51 /mimic/demo/raw
drwxr-xr-x   - root supergroup          0 2026-10-06 07:51 /mimic/demo/raw/hosp
-rw-r--r--   1 root supergroup      11072 2026-10-06 07:51 /mimic/demo/raw/hosp/admissions.csv.gz
-rw-r--r--   1 root supergroup     510070 2026-10-06 07:51 /mimic/demo/raw/hosp/d_hcpcs.csv.gz
-rw-r--r--   1 root supergroup    1813533 2026-10-06 07:51 /mimic/demo/raw/hosp/d_icd_diagnoses.csv.gz


9.1 Контрольная проверка (итоговая)

In [24]:
# Чек-лист защиты
checks = {
    "Дерево /mimic/demo показано": True,
    "fsck выполнен": "HEALTHY" in hdfs("-fsck /mimic/demo/raw/hosp/patients.csv.gz -files -blocks"),
    "SHA256 совпал": ok_count == len(results),
    "Python-скрипт работает": len(results) > 0,
    "Все 33 файла проверены": len(results) == 34,
}

print("=== ЧЕК-ЛИСТ ЗАЩИТЫ ===")
for item, status in checks.items():
    print(f"{'✅' if status else '❌'} {item}")

total = sum(checks.values())
print(f"\nИтого: {total}/{len(checks)} пунктов выполнено")

⚠️ Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&path=%2Fmimic%2Fdemo%2Fraw%2Fhosp%2Fpatients.csv.gz

=== ЧЕК-ЛИСТ ЗАЩИТЫ ===
✅ Дерево /mimic/demo показано
✅ fsck выполнен
✅ SHA256 совпал
✅ Python-скрипт работает
✅ Все 33 файла проверены

Итого: 5/5 пунктов выполнено


Вопросы для защиты:

Почему MIMIC-IV разделён на hosp/ и icu/? — Две разные схемы: госпитальные данные и данные ОРИТ.

Что общего между subject_id и путём в HDFS? — Оба однозначно идентифицируют объект.

Риски хранения клинических данных? — Утечка, потеря, несанкционированный доступ.

Почему нельзя всё в одну папку? — Теряется организация, сложно искать, нет разграничения прав.